# Aula 3 • Desafio Investigativo — O caso da venda impossível

## NumPy, Pandas, estatística descritiva, Matplotlib e Seaborn

A **Loja Conecta** enviou uma base de vendas para análise. A diretoria desconfia de uma venda muito acima do padrão e quer saber se o relatório pode ser usado para tomar decisões.

Sua missão é investigar os dados e produzir um relatório curto, sustentado por código e evidências.

### Entrega final

Você deverá apresentar três achados:

1. **Qualidade:** um problema encontrado e a decisão tomada;
2. **Comparação:** uma diferença entre categorias, regiões ou canais;
3. **Relação:** uma associação entre duas variáveis numéricas.

Cada achado deve conter **pergunta + código + evidência + conclusão + limitação**.

> Execute as células na ordem. Não pule o diagnóstico para chegar aos gráficos.


## Mapa da missão

| Etapa | O que você fará | Evidência de conclusão |
|---|---|---|
| 0 | Preparar a base | arquivo criado e bibliotecas carregadas |
| 1 | Reconhecer os dados | estrutura e tipos explicados |
| 2 | Usar NumPy | cálculo vetorizado sem laço |
| 3 | Investigar a qualidade | ausentes, duplicatas e inconsistências localizados |
| 4 | Limpar com justificativa | base limpa e decisão sobre o extremo |
| 5 | Resumir | centro, dispersão, quartis e agrupamentos |
| 6 | Visualizar | gráficos adequados às perguntas |
| 7 | Comunicar | relatório de três achados |




## 0. Preparação automática da base

Execute a próxima célula **sem alterá-la**. Ela cria uma base fictícia e reproduzível com problemas intencionais.

Os dados são sintéticos e usados somente para fins didáticos.


In [2]:
# Execute sem alterar
import numpy as np4
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

rng = np.random.default_rng(42)
n = 48

datas = pd.date_range("2026-03-01", periods=n, freq="D")
categorias = rng.choice(["Tecnologia", "Casa", "Papelaria", "Esportes"], n,
                        p=[0.28, 0.28, 0.24, 0.20])
regioes = rng.choice(["Sudeste", "Sul", "Nordeste", "Centro-Oeste"], n,
                     p=[0.40, 0.25, 0.23, 0.12])
canais = rng.choice(["Site", "App", "Loja"], n, p=[0.45, 0.35, 0.20])
unidades = rng.integers(1, 8, n)

preco_base = pd.Series(categorias).map({
    "Tecnologia": 340,
    "Casa": 155,
    "Papelaria": 48,
    "Esportes": 210,
}).to_numpy()

preco_unitario = np.round(preco_base * rng.normal(1, 0.16, n), 2).clip(18)
desconto = rng.choice([0, 0.05, 0.10, 0.15], n, p=[0.35, 0.30, 0.23, 0.12])
avaliacao = np.round(rng.normal(4.2, 0.50, n), 1).clip(1, 5)
vendas = np.round(unidades * preco_unitario * (1 - desconto), 2)

base = pd.DataFrame({
    "data": datas,
    "categoria": categorias,
    "regiao": regioes,
    "canal": canais,
    "unidades": unidades,
    "preco_unitario": preco_unitario,
    "desconto": desconto,
    "avaliacao": avaliacao,
    "vendas": vendas,
})

# Problemas intencionais
base.loc[[6, 31], "regiao"] = np.nan
base.loc[[8, 27, 35], "avaliacao"] = np.nan
base.loc[10, "categoria"] = " tecnologia "
base.loc[20, "canal"] = "APP"
base.loc[12, "vendas"] = np.round(base.loc[12, "vendas"] * 5, 2)
base = pd.concat([base, base.iloc[[4, 18]]], ignore_index=True)

base.to_csv("vendas_desafio_aula3.csv", index=False)
print("Arquivo criado: vendas_desafio_aula3.csv")


Arquivo criado: vendas_desafio_aula3.csv


## 1. Reconhecer antes de transformar

### Passo 1.1 — Carregue a base

Use `pd.read_csv()`, guarde o resultado em `df` e mostre as cinco primeiras linhas.


In [3]:
df = pd.read_csv("vendas_desafio_aula3.csv")
df.head()

,data,categoria,regiao,canal,unidades,preco_unitario,desconto,avaliacao,vendas
0,2026-03-01,Papelaria,Nordeste,Site,4,41.07,0.00,4.4,164.28
1,2026-03-02,Casa,Sudeste,Site,4,178.10,0.15,4.0,605.54
2,2026-03-03,Esportes,Sudeste,Site,6,222.93,0.05,4.1,1270.70
3,2026-03-04,Papelaria,Sudeste,Loja,2,46.80,0.00,3.5,93.60
4,2026-03-05,Tecnologia,Nordeste,Loja,7,337.78,0.05,4.1,2246.24


In [ ]:
assert isinstance(df, pd.DataFrame), "df ainda não é um DataFrame. Use pd.read_csv()."
assert df.shape == (50, 9), "A base deve ter 50 linhas e 9 colunas."
print("✅ Base carregada corretamente.")


✅ Base carregada corretamente.


### Passo 1.2 — Faça o inventário inicial

Descubra:

- quantidade de linhas e colunas;
- nomes das colunas;
- tipos de dados;
- resumo produzido por `df.info()`.

Depois responda: **qual coluna deveria ser data, mas ainda não foi reconhecida como data?**


In [ ]:
print("shape:\n", df.shape)
print("colunas:\n", df.columns)
print("dtypes:\n", df.dtypes)
print("infos:", end='\n\n')
df.info()

shape:
           data     categoria    regiao canal  unidades  preco_unitario  \
0   2026-03-01     Papelaria  Nordeste  Site         4           41.07   
1   2026-03-02          Casa   Sudeste  Site         4          178.10   
2   2026-03-03      Esportes   Sudeste  Site         6          222.93   
3   2026-03-04     Papelaria   Sudeste  Loja         2           46.80   
4   2026-03-05    Tecnologia  Nordeste  Loja         7          337.78   
5   2026-03-06      Esportes  Nordeste   App         3          188.00   
6   2026-03-07     Papelaria       NaN  Site         4           51.43   
7   2026-03-08     Papelaria  Nordeste  Loja         4           44.51   
8   2026-03-09    Tecnologia       Sul   App         6          273.33   
9   2026-03-10          Casa       Sul   App         4          123.31   
10  2026-03-11   tecnologia    Sudeste  Site         2          159.28   
11  2026-03-12      Esportes   Sudeste  Site         1          263.06   
12  2026-03-13     Papelaria  

TypeError: 'Index' object is not callable

**Sua resposta:** escreva aqui qual coluna precisa de conversão e por quê.


## 2. Aquecimento NumPy — calcule sem laço

Converta as colunas `unidades`, `preco_unitario` e `desconto` para arrays NumPy.

Calcule, de forma vetorizada:

[
	ext{venda esperada} = 	ext{unidades} 	imes 	ext{preço unitário} 	imes (1 - 	ext{desconto})
]

Guarde o resultado em `venda_esperada` e mostre os cinco primeiros valores.


In [ ]:
# TODO: crie três arrays com to_numpy()
# TODO: calcule venda_esperada sem usar for
venda_esperada = None


In [ ]:
assert isinstance(venda_esperada, np.ndarray), "O resultado deve ser um array NumPy."
assert len(venda_esperada) == len(df), "O array precisa ter um valor por linha."
assert np.isclose(venda_esperada[0], df.loc[0, "unidades"] * df.loc[0, "preco_unitario"] * (1 - df.loc[0, "desconto"])), "Revise a fórmula."
print("✅ Cálculo vetorizado correto.")


### Passo 2.1 — Localize divergências

Compare `venda_esperada` com a coluna `vendas`. Crie a coluna `diferenca_venda` com o valor absoluto da diferença e ordene da maior para a menor.

**Pergunta:** qual linha merece investigação?


In [ ]:
# TODO: crie diferenca_venda
# TODO: mostre as cinco maiores diferenças



**Sua resposta:** registre o índice encontrado e explique por que a linha merece investigação.


## 3. Auditoria da qualidade

### Passo 3.1 — Conte os problemas

Calcule:

- valores ausentes por coluna;
- linhas exatamente duplicadas;
- valores únicos de `categoria` e `canal`.

Procure diferenças de espaços, maiúsculas e minúsculas.


In [ ]:
# TODO: use isna(), duplicated() e unique()



### Passo 3.2 — Registre o diagnóstico

Preencha antes de limpar:

| Problema | Onde apareceu? | Risco para a análise | Decisão proposta |
|---|---|---|---|
| Ausentes | ... | ... | ... |
| Duplicatas | ... | ... | ... |
| Texto inconsistente | ... | ... | ... |
| Venda extrema | ... | ... | ... |

> A transformação só deve acontecer depois que sua decisão puder ser explicada.


## 4. Limpeza com justificativa

Crie `df_limpo` como cópia de `df` e aplique estas decisões:

1. converter `data` para `datetime`;
2. remover espaços e padronizar categoria e canal;
3. preencher `regiao` com a moda;
4. preencher `avaliacao` com a mediana;
5. remover duplicatas exatas;
6. **não apagar automaticamente** a venda suspeita: criar a coluna booleana `venda_suspeita` usando `diferenca_venda > 0.01`.

Escreva um comentário em cada operação justificando a decisão.


In [ ]:
# TODO: crie df_limpo e aplique as seis decisões
df_limpo = None


In [ ]:
assert isinstance(df_limpo, pd.DataFrame), "Crie df_limpo como DataFrame."
assert df_limpo.shape[0] == 48, "Depois de remover duplicatas, devem restar 48 linhas."
assert df_limpo.isna().sum().sum() == 0, "Ainda existem valores ausentes."
assert pd.api.types.is_datetime64_any_dtype(df_limpo["data"]), "Converta data para datetime."
assert set(df_limpo["categoria"]) == {"Tecnologia", "Casa", "Papelaria", "Esportes"}, "Revise a padronização de categoria."
assert set(df_limpo["canal"]) == {"Site", "App", "Loja"}, "Revise a padronização de canal."
assert df_limpo["venda_suspeita"].sum() == 1, "Deve existir exatamente uma venda sinalizada."
print("✅ Limpeza concluída e venda suspeita preservada para investigação.")


### Passo 4.1 — Confirme o extremo pelo IQR

Calcule `Q1`, `Q3`, `IQR`, limite inferior e limite superior para `vendas`. Liste os registros fora dos limites.

Lembre-se: o critério de 1,5 × IQR **sinaliza** pontos; ele não determina automaticamente a remoção.


In [ ]:
# TODO: calcule Q1, Q3, IQR e os limites
# TODO: liste os registros fora dos limites



## 5. Estatística descritiva — resumir sem esconder

### Passo 5.1 — Centro e dispersão

Para `vendas`, calcule:

- média e mediana;
- mínimo, máximo e amplitude;
- variância e desvio-padrão amostrais;
- Q1, Q3, IQR e percentil 90.

Guarde os resultados em uma `Series` chamada `resumo_vendas`.


In [ ]:
# TODO: crie resumo_vendas
resumo_vendas = None


In [ ]:
assert isinstance(resumo_vendas, pd.Series), "Use uma Series para organizar o resumo."
assert len(resumo_vendas) >= 10, "O resumo está incompleto."
assert resumo_vendas["máximo"] > resumo_vendas["percentil 90"], "Revise máximo e percentil."
print("✅ Resumo estatístico construído.")


### Passo 5.2 — A venda suspeita muda a história?

Compare média e mediana:

- usando todas as vendas;
- sem a linha marcada como suspeita.

Explique qual medida foi mais afetada e por quê.


In [ ]:
# TODO: compare média e mediana com e sem a venda suspeita



**Sua interpretação:** qual medida mudou mais? Relacione sua resposta à definição de média e mediana.


### Passo 5.3 — Compare grupos com `groupby`

Crie `resumo_categoria` com uma linha por categoria e as colunas:

- `vendas_total`;
- `ticket_mediano`;
- `avaliacao_media`;
- `quantidade_vendas`.

Ordene da maior para a menor venda total.


In [ ]:
# TODO: use groupby().agg() e sort_values()
resumo_categoria = None


In [ ]:
assert isinstance(resumo_categoria, pd.DataFrame), "Crie resumo_categoria como DataFrame."
assert resumo_categoria.shape == (4, 4), "São esperadas 4 categorias e 4 métricas."
assert resumo_categoria["vendas_total"].is_monotonic_decreasing, "Ordene vendas_total de forma decrescente."
print("✅ Comparação por categoria concluída.")


## 6. Visualização — escolha o gráfico pela pergunta

Crie pelo menos **três** gráficos. Todos devem ter título, rótulos dos eixos e tamanho legível.

### Gráfico 1 — Comparar

Use Matplotlib para criar um gráfico de barras com vendas totais por categoria.


In [ ]:
# TODO: gráfico de barras com Matplotlib



### Gráfico 2 — Distribuir e detectar extremos

Crie:

- um histograma de `vendas` com Seaborn;
- um boxplot de `vendas` por categoria.

Observe concentração, assimetria e pontos extremos.


In [ ]:
# TODO: histograma e boxplot



### Gráfico 3 — Relacionar

Crie um gráfico de dispersão entre `unidades` e `vendas`. Use a categoria como cor.

Depois calcule a correlação entre as variáveis numéricas.

> Correlação organiza pistas; não prova causalidade.


In [ ]:
# TODO: scatterplot e matriz de correlação



### Desafio bônus — mapa de calor

Represente `correlacoes` com `sns.heatmap()`. Use `annot=True`, uma escala de -1 a 1 e um mapa de cores divergente.


In [ ]:
# BÔNUS: crie o heatmap



## 7. Relatório final — três achados

Complete cada bloco em três a cinco frases. Evite frases vagas como “o gráfico ficou alto”. Use números, grupos e variáveis.

### Achado 1 — Qualidade

- **Pergunta:** qual problema poderia distorcer a análise?
- **Evidência:** indique contagem, linha ou diferença observada.
- **Decisão:** explique o que você fez.
- **Conclusão:** diga por que a base ficou mais confiável.
- **Limitação:** o que ainda não pode ser confirmado apenas com estes dados?

**Sua resposta:**

...

### Achado 2 — Comparação

- **Pergunta:** qual categoria, região ou canal se destaca?
- **Evidência:** cite uma métrica do agrupamento e um gráfico.
- **Conclusão:** descreva a diferença encontrada.
- **Limitação:** volume total, ticket e quantidade de registros contam a mesma história?

**Sua resposta:**

...

### Achado 3 — Relação

- **Pergunta:** duas variáveis numéricas parecem variar juntas?
- **Evidência:** cite o gráfico de dispersão e a correlação.
- **Conclusão:** descreva apenas a associação observada.
- **Limitação:** explique por que correlação não demonstra causa.

**Sua resposta:**

...


## Checklist antes da entrega

- [ ] Executei todas as células na ordem.
- [ ] Mantive a base bruta e criei `df_limpo` separadamente.
- [ ] Justifiquei cada decisão de limpeza.
- [ ] Não removi o extremo apenas porque apareceu no boxplot.
- [ ] Calculei centro, dispersão, quartis e percentil.
- [ ] Usei `groupby` para comparar grupos.
- [ ] Todos os gráficos têm título e rótulos.
- [ ] Minhas conclusões citam evidências.
- [ ] Reconheci pelo menos uma limitação.

> Dados não falam sozinhos: nós fazemos perguntas, verificamos evidências e comunicamos limites.
